# Sesión 5 — SQL de negocio y análisis operativo

**Mensaje central:** Silver nos da datos confiables; SQL nos permite convertirlos en evidencia, insights y decisiones.

En esta sesión trabajaremos sobre las tablas Silver creadas en la Sesión 4. El objetivo no es crear Gold todavía, sino dejar consultas candidatas y entender bien granularidad, joins, agregaciones y riesgo de doble conteo.

## 0. Contexto técnico esperado

Catálogo: `workspace`

Tablas principales:

- `workspace.lumi_silver.orders_clean`
- `workspace.lumi_silver.order_items_clean`
- `workspace.lumi_silver.payments_clean`
- `workspace.lumi_silver.reviews_clean`
- `workspace.lumi_silver.products_clean`
- `workspace.lumi_silver.sellers_clean`
- `workspace.bagazo_silver.operacion_ingenios_clean`
- `workspace.control.quality_summary_sesion_04`

In [0]:
%sql
USE CATALOG workspace;
SHOW SCHEMAS;

## 1. Salud de Silver

Abrimos la clase con gobierno de datos. Antes de analizar, revisamos qué tan confiables son las tablas que vamos a consultar.

In [0]:
%sql
SELECT *
FROM workspace.control.quality_summary_sesion_04
ORDER BY dataset, tabla;

In [0]:
%sql
SELECT
  estado_calidad,
  COUNT(*) AS tablas
FROM workspace.control.quality_summary_sesion_04
GROUP BY estado_calidad
ORDER BY tablas DESC;

In [0]:
%sql
SELECT
  dataset, tabla, filas, columnas, duplicados_clave, reglas_fallidas, estado_calidad, observaciones
FROM workspace.control.quality_summary_sesion_04
WHERE estado_calidad <> 'OK'
ORDER BY dataset, tabla;

### Reflexión guiada

`reviews_clean` quedó en estado **REVISAR** por duplicados de clave. Esto no bloquea el análisis, pero sí cambia la forma correcta de hacer joins. Cuando crucemos reviews con pedidos o ítems, primero agregaremos reviews por `order_id`.

## 2. Granularidad: la regla que evita métricas infladas

- `orders_clean`: una fila por pedido.
- `payments_clean`: consolidada por `order_id`.
- `order_items_clean`: una fila por ítem de pedido.
- `reviews_clean`: puede tener más de una fila por pedido; requiere agregación previa.
- `operacion_ingenios_clean`: una fila por fecha e ingenio.

**Regla de oro:** si dos tablas tienen granularidad diferente, agrega primero y une después.

# Bloque A — Lumi Commerce Lakehouse

## 3. Ventas por mes

Usamos `order_items_clean` porque la venta está a nivel de ítem. Unimos con `orders_clean` para filtrar pedidos entregados y agrupar por mes.

**TODO pedagógico 1:** cambia temporalmente el filtro `order_status = 'delivered'` por otro estado y compara el resultado.

In [0]:
%sql
WITH ventas_item AS (
  SELECT
    o.year_month,
    oi.order_id,
    oi.order_item_id,
    oi.total_item_value
  FROM workspace.lumi_silver.order_items_clean oi
  INNER JOIN workspace.lumi_silver.orders_clean o
    ON oi.order_id = o.order_id
  WHERE o.order_status = 'delivered'
)
SELECT
  year_month,
  COUNT(DISTINCT order_id) AS pedidos_entregados,
  COUNT(*) AS items_vendidos,
  ROUND(SUM(total_item_value), 2) AS venta_total_items,
  ROUND(AVG(total_item_value), 2) AS valor_promedio_item
FROM ventas_item
GROUP BY year_month
ORDER BY year_month;

## 4. Ticket promedio por pedido

Aquí usamos `payments_clean`, que ya quedó consolidada por pedido en Silver. Por eso podemos calcular ticket promedio sin unir contra ítems.

**TODO pedagógico 2:** identifica por qué usamos `COUNT(DISTINCT o.order_id)` y no simplemente `COUNT(*)` en análisis con joins.

In [0]:
%sql
SELECT
  o.year_month,
  COUNT(DISTINCT o.order_id) AS pedidos_entregados,
  ROUND(SUM(COALESCE(p.total_payment_value, 0)), 2) AS valor_pagado_total,
  ROUND(AVG(COALESCE(p.total_payment_value, 0)), 2) AS ticket_promedio_pedido
FROM workspace.lumi_silver.orders_clean o
LEFT JOIN workspace.lumi_silver.payments_clean p
  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY o.year_month
ORDER BY o.year_month;

## 5. Pedidos por estado

In [0]:
%sql
SELECT
  order_status,
  COUNT(*) AS pedidos,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM workspace.lumi_silver.orders_clean
GROUP BY order_status
ORDER BY pedidos DESC;

## 6. Ventas por categoría

Cuidamos la granularidad: ventas por categoría se calcula desde ítems, no desde pagos.

In [0]:
%sql
WITH ventas_categoria AS (
  SELECT
    COALESCE(p.product_category_clean, 'sin_categoria') AS categoria,
    oi.order_id,
    oi.order_item_id,
    oi.total_item_value
  FROM workspace.lumi_silver.order_items_clean oi
  INNER JOIN workspace.lumi_silver.orders_clean o
    ON oi.order_id = o.order_id
  LEFT JOIN workspace.lumi_silver.products_clean p
    ON oi.product_id = p.product_id
  WHERE o.order_status = 'delivered'
)
SELECT
  categoria,
  COUNT(DISTINCT order_id) AS pedidos,
  COUNT(*) AS items,
  ROUND(SUM(total_item_value), 2) AS venta_total,
  ROUND(AVG(total_item_value), 2) AS valor_promedio_item
FROM ventas_categoria
GROUP BY categoria
ORDER BY venta_total DESC
LIMIT 20;

## 7. Métodos de pago

In [0]:
%sql
SELECT
  main_payment_type,
  COUNT(*) AS pedidos,
  ROUND(SUM(total_payment_value), 2) AS valor_total_pagado,
  ROUND(AVG(total_payment_value), 2) AS ticket_promedio
FROM workspace.lumi_silver.payments_clean
GROUP BY main_payment_type
ORDER BY valor_total_pagado DESC;

## 8. Reviews por categoría

Antes de cruzar reviews con categorías, agregamos reviews por pedido.

**TODO pedagógico 3:** en la CTE `reviews_by_order`, identifica la métrica que representa satisfacción promedio por pedido.

In [0]:
%sql
WITH reviews_by_order AS (
  SELECT
    order_id,
    ROUND(AVG(review_score), 2) AS review_promedio_pedido,
    MAX(CASE WHEN is_low_review THEN 1 ELSE 0 END) AS tiene_review_bajo
  FROM workspace.lumi_silver.reviews_clean
  GROUP BY order_id
),
category_orders AS (
  SELECT DISTINCT
    oi.order_id,
    COALESCE(p.product_category_clean, 'sin_categoria') AS categoria
  FROM workspace.lumi_silver.order_items_clean oi
  LEFT JOIN workspace.lumi_silver.products_clean p
    ON oi.product_id = p.product_id
)
SELECT
  co.categoria,
  COUNT(DISTINCT co.order_id) AS pedidos_con_categoria,
  ROUND(AVG(r.review_promedio_pedido), 2) AS review_promedio,
  SUM(CASE WHEN r.tiene_review_bajo = 1 THEN 1 ELSE 0 END) AS pedidos_con_review_bajo
FROM category_orders co
LEFT JOIN reviews_by_order r
  ON co.order_id = r.order_id
GROUP BY co.categoria
HAVING COUNT(DISTINCT co.order_id) >= 50
ORDER BY review_promedio ASC, pedidos_con_categoria DESC
LIMIT 20;

## 9. Entregas tardías

In [0]:
%sql
SELECT
  year_month,
  COUNT(*) AS pedidos_entregados,
  SUM(CASE WHEN is_late THEN 1 ELSE 0 END) AS pedidos_tarde,
  ROUND(100.0 * SUM(CASE WHEN is_late THEN 1 ELSE 0 END) / COUNT(*), 2) AS tasa_entrega_tarde,
  ROUND(AVG(delay_days), 2) AS demora_promedio_dias
FROM workspace.lumi_silver.orders_clean
WHERE order_status = 'delivered'
GROUP BY year_month
ORDER BY year_month;

## 10. Relación entre demora y review

In [0]:
%sql
WITH reviews_by_order AS (
  SELECT
    order_id,
    ROUND(AVG(review_score), 2) AS review_promedio_pedido
  FROM workspace.lumi_silver.reviews_clean
  GROUP BY order_id
)
SELECT
  CASE
    WHEN o.delay_days <= 0 THEN 'A tiempo o antes'
    WHEN o.delay_days BETWEEN 1 AND 3 THEN '1 a 3 días tarde'
    WHEN o.delay_days BETWEEN 4 AND 7 THEN '4 a 7 días tarde'
    ELSE 'Más de 7 días tarde'
  END AS tramo_demora,
  COUNT(DISTINCT o.order_id) AS pedidos,
  ROUND(AVG(o.delay_days), 2) AS demora_promedio,
  ROUND(AVG(r.review_promedio_pedido), 2) AS review_promedio
FROM workspace.lumi_silver.orders_clean o
LEFT JOIN reviews_by_order r
  ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY tramo_demora
ORDER BY demora_promedio;

## 11. Ranking de vendedores

In [0]:
%sql
SELECT
  s.seller_id,
  s.seller_state,
  COUNT(DISTINCT oi.order_id) AS pedidos,
  COUNT(*) AS items,
  ROUND(SUM(oi.total_item_value), 2) AS venta_total,
  ROUND(AVG(oi.total_item_value), 2) AS valor_promedio_item
FROM workspace.lumi_silver.order_items_clean oi
INNER JOIN workspace.lumi_silver.orders_clean o
  ON oi.order_id = o.order_id
LEFT JOIN workspace.lumi_silver.sellers_clean s
  ON oi.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
GROUP BY s.seller_id, s.seller_state
ORDER BY venta_total DESC
LIMIT 20;

## 12. Categorías con alta venta y baja satisfacción

In [0]:
%sql
WITH ventas_categoria AS (
  SELECT
    COALESCE(p.product_category_clean, 'sin_categoria') AS categoria,
    COUNT(DISTINCT oi.order_id) AS pedidos,
    ROUND(SUM(oi.total_item_value), 2) AS venta_total
  FROM workspace.lumi_silver.order_items_clean oi
  INNER JOIN workspace.lumi_silver.orders_clean o
    ON oi.order_id = o.order_id
  LEFT JOIN workspace.lumi_silver.products_clean p
    ON oi.product_id = p.product_id
  WHERE o.order_status = 'delivered'
  GROUP BY COALESCE(p.product_category_clean, 'sin_categoria')
),
reviews_by_order AS (
  SELECT order_id, AVG(review_score) AS review_promedio_pedido
  FROM workspace.lumi_silver.reviews_clean
  GROUP BY order_id
),
category_orders AS (
  SELECT DISTINCT
    oi.order_id,
    COALESCE(p.product_category_clean, 'sin_categoria') AS categoria
  FROM workspace.lumi_silver.order_items_clean oi
  LEFT JOIN workspace.lumi_silver.products_clean p
    ON oi.product_id = p.product_id
),
reviews_categoria AS (
  SELECT
    co.categoria,
    ROUND(AVG(r.review_promedio_pedido), 2) AS review_promedio
  FROM category_orders co
  LEFT JOIN reviews_by_order r
    ON co.order_id = r.order_id
  GROUP BY co.categoria
)
SELECT
  v.categoria,
  v.pedidos,
  v.venta_total,
  r.review_promedio,
  CASE
    WHEN v.venta_total >= 100000 AND r.review_promedio < 4.0 THEN 'Alta prioridad'
    WHEN v.venta_total >= 50000 AND r.review_promedio < 4.0 THEN 'Revisar'
    ELSE 'Monitorear'
  END AS prioridad_analitica
FROM ventas_categoria v
LEFT JOIN reviews_categoria r
  ON v.categoria = r.categoria
WHERE v.pedidos >= 100
ORDER BY prioridad_analitica, v.venta_total DESC, r.review_promedio ASC
LIMIT 25;

# Bloque B — Lluvia, caña y bagazo

## 13. Lectura operativa del caso Bagazo

La tabla quedó en granularidad limpia: una fila por `fecha` e `ingenio`. Aquí los ceros operativos no se tratan automáticamente como errores: se interpretan con comentarios y banderas.

In [0]:
%sql
SELECT *
FROM workspace.bagazo_silver.operacion_ingenios_clean
LIMIT 20;

## 14. Lluvia y bagazo por mes

In [0]:
%sql
SELECT
  year_month,
  ROUND(AVG(lluvia_mm), 2) AS lluvia_promedio_mm,
  ROUND(AVG(bagazo_entregado_ton), 2) AS bagazo_promedio_ton,
  ROUND(AVG(cana_molida_ton), 2) AS cana_promedio_ton,
  COUNT(*) AS registros
FROM workspace.bagazo_silver.operacion_ingenios_clean
GROUP BY year_month
ORDER BY year_month;

## 15. Caña y bagazo por ingenio

In [0]:
%sql
SELECT
  ingenio,
  ROUND(SUM(cana_molida_ton), 2) AS cana_total_ton,
  ROUND(SUM(bagazo_entregado_ton), 2) AS bagazo_total_ton,
  ROUND(AVG(bagazo_entregado_ton), 2) AS bagazo_promedio_ton,
  SUM(CASE WHEN riesgo_bajo_bagazo THEN 1 ELSE 0 END) AS dias_riesgo_bajo
FROM workspace.bagazo_silver.operacion_ingenios_clean
GROUP BY ingenio
ORDER BY bagazo_total_ton DESC;

## 16. Días con lluvia alta y bajo bagazo

In [0]:
%sql
SELECT
  fecha, ingenio, lluvia_mm, cana_molida_ton, bagazo_entregado_ton,
  lluvia_alta, riesgo_bajo_bagazo, tiene_comentario_operativo, comentario
FROM workspace.bagazo_silver.operacion_ingenios_clean
WHERE lluvia_alta = true OR riesgo_bajo_bagazo = true
ORDER BY fecha, ingenio
LIMIT 80;

## 17. Promedio de bagazo en días secos vs lluviosos

In [0]:
%sql
SELECT
  ingenio,
  CASE WHEN lluvia_alta THEN 'Día con lluvia alta' ELSE 'Día sin lluvia alta' END AS tipo_dia,
  COUNT(*) AS dias,
  ROUND(AVG(lluvia_mm), 2) AS lluvia_promedio_mm,
  ROUND(AVG(cana_molida_ton), 2) AS cana_promedio_ton,
  ROUND(AVG(bagazo_entregado_ton), 2) AS bagazo_promedio_ton
FROM workspace.bagazo_silver.operacion_ingenios_clean
GROUP BY ingenio, CASE WHEN lluvia_alta THEN 'Día con lluvia alta' ELSE 'Día sin lluvia alta' END
ORDER BY ingenio, tipo_dia;

## 18. Correlación simple por ingenio

In [0]:
%sql
SELECT
  ingenio,
  COUNT(*) AS observaciones,
  ROUND(CORR(lluvia_mm, bagazo_entregado_ton), 4) AS corr_lluvia_bagazo,
  ROUND(CORR(cana_molida_ton, bagazo_entregado_ton), 4) AS corr_cana_bagazo,
  ROUND(CORR(lluvia_mm, cana_molida_ton), 4) AS corr_lluvia_cana
FROM workspace.bagazo_silver.operacion_ingenios_clean
GROUP BY ingenio
ORDER BY ingenio;

## 19. Días críticos explicados por comentarios operativos

In [0]:
%sql
SELECT
  fecha, ingenio, lluvia_mm, cana_molida_ton, bagazo_entregado_ton,
  es_mantenimiento, es_falta_cana_por_lluvia, es_paro, es_sin_recepcion_bagazo,
  comentario
FROM workspace.bagazo_silver.operacion_ingenios_clean
WHERE riesgo_bajo_bagazo = true
  AND tiene_comentario_operativo = true
ORDER BY fecha, ingenio
LIMIT 50;

# Cierre — De SQL a insight ejecutivo

**TODO pedagógico 4:** escribe un insight ejecutivo usando este formato:

```text
Insight:
Evidencia:
Impacto operativo o de negocio:
Recomendación:
Consulta SQL usada:
```

# Retos finales

Los retos no bloquean el flujo principal. Úsalos para práctica individual o trabajo por grupos.

## Reto nivel 1
Identifica las 10 categorías con mayor review promedio o los 10 estados con más pedidos entregados.

## Reto nivel 2
Construye una CTE que combine ventas, demora y review promedio por categoría sin duplicar pagos ni pedidos.

## Reto consultor
Entrega 3 insights en formato ejecutivo, cada uno respaldado por una consulta SQL.